1.Загрузка изображения и преобразование цветного к полутоновому

In [ ]:
import marimo as mo
import matplotlib.pyplot as plt
import numpy as np
import cv2

In [ ]:
def show(img, title=""):
    fig, ax = plt.subplots(figsize=(5, 7))
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    plt.show()

In [ ]:
image = cv2.imread("image.jpg", cv2.IMREAD_COLOR_RGB)
gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

show(gray)

> **Теперь я строю гистограмму яркости.** Функция `ravel` разворачивает двумерное изображение в одномерный массив, а `np.bincount` считает, сколько раз встретилось каждое значение от 0 до 255. Поэтому по горизонтальной оси находится яркость, а по вертикальной — количество пикселей с этой яркостью. **По форме гистограммы можно понять, какие тона преобладают, и затем выбирать пороги для разделения изображения.**

In [ ]:
def histogram(img):
    return np.bincount(img.ravel(), minlength=256)


hist = histogram(gray)

_fig, _ax = plt.subplots(figsize=(12, 3))
_ax.bar(np.arange(256), hist, width=1)
plt.show()

> **На гистограмме преобладают тёмные оттенки, а самый выраженный пик находится примерно между 30 и 60.** При этом а в более светлых диапазонах количество пикселей заметно меньше и распределено шире. Это соответствует самой фотографии, где тёмные области преобладают, а светлые занимают меньшую часть кадра. Такая гистограмма объясняет, почему выбор порога и способ деления диапазона яркостей существенно влияют на итоговое изображение.

> **Здесь я реализую квантование — уменьшение числа допустимых уровней яркости.** Вместо того чтобы обрабатывать каждый пиксель отдельно, создаю таблицу соответствий `lut` сразу для всех 256 исходных значений. Сначала целочисленным делением распределяю яркости по заданному числу групп, затем перевожу номера групп обратно в диапазон от 0 до 255. `cv2.LUT` заменяет яркость каждого пикселя по этой таблице. Размер изображения при этом не меняется, меняется только число возможных значений яркости. Функция рассчитана на `levels` не меньше двух.

In [ ]:
def quantize(img, levels):
    lut = (np.arange(256) * levels // 256) * 255 // (levels - 1)
    return cv2.LUT(img, lut.astype(np.uint8))

2.Квантование по яркости (с ручным заданием количества уровней)

> **Я последовательно применяю квантование с двумя, четырьмя, восемью и шестнадцатью уровнями.** Для каждого варианта использую одну и ту же исходную полутоновую картинку, поэтому результаты можно сравнивать напрямую. При двух уровнях изображение содержит только чёрный и белый, а с увеличением количества уровней появляются промежуточные оттенки и обычно сохраняется больше деталей. Здесь границы уровней назначены равномерно, а не подбираются по содержимому картинки.

In [ ]:
for levels in [2, 4, 8, 16]:
    Q = quantize(gray, levels)
    show(Q, f"levels = {levels}")

> *Вывод по четырём изображениям.* При 2 уровнях большая часть полутонов исчезает: остаются резкие чёрно-белые области, а детали лица, волос и одежды частично теряются. При 4 уровнях основные формы и разделение фона становятся заметнее, при 8 уровнях появляется больше объёма и фактуры, а при 16 уровнях изображение уже заметно ближе к исходному серому. **Следовательно, увеличение числа уровней позволяет сохранить больше градаций яркости, но уменьшает степень упрощения изображения.** Здесь количество уровней задавал я, без анализа содержимого фотографии.

Далее я перехожу от квантования к бинаризации. **Бинаризация оставляет только два значения пикселя — 0 и 255 — и делит изображение на тёмную и светлую части.** Сначала исследую глобальный метод Оцу: он ищет один порог по гистограмме всего изображения, без ручного выбора этого значения.

3. Бинаризация
    1. Бинаризация методом Оцу глобальная

> Здесь я самостоятельно реализую метод Оцу. Сначала беру гистограмму в заданном диапазоне яркостей и превращаю количества пикселей в вероятности, разделив на общее число пикселей. Затем `np.cumsum` помогает для каждого возможного порога посчитать долю пикселей первого класса и накопленную сумму их яркостей. **По этим величинам вычисляю межклассовую дисперсию — показатель того, насколько хорошо порог отделяет две группы с разной средней яркостью. Выбираю порог, при котором этот показатель максимален.** Некорректные значения на крайних порогах заменяю нулями, чтобы они не мешали поиску максимума.

In [ ]:
def otsu(hist, lo=0, hi=256):
    h = hist[lo:hi].astype(np.float64)
    p = h / h.sum()
    levels = np.arange(lo, hi)

    w = np.cumsum(p)
    mu = np.cumsum(p * levels)
    mu_t = mu[-1]

    with np.errstate(divide="ignore", invalid="ignore"):
        sigma_b = (mu_t * w - mu) ** 2 / (w * (1 - w))
    sigma_b = np.nan_to_num(sigma_b[:-1])

    k = int(np.argmax(sigma_b))
    return lo + k, sigma_b[k]

> Сейчас я **проверяю правильность своего метода Оцу сравнением с OpenCV**. Сначала моя функция возвращает порог `t`, затем `cv2.threshold` с флагом `THRESH_OTSU` автоматически вычисляет порог `t_cv`. Вывожу оба значения на экран. Для этой фотографии они совпали — оба равны 97. Это означает, что мой алгоритм поиска порога согласуется с библиотечным на данном входном изображении.

In [ ]:
t, _ = otsu(hist)
t_cv, _ = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(t, t_cv)

> *Вывод по числам.* В ноутбуке напечатано `97 97.0`. **Моя реализация и OpenCV совпали — обе нашли порог 97.** Это проверяет, что обе реализации согласованы для данной фотографии. Число 97 — не вручную выбранная граница, а результат анализа исходной гистограммы.

После выбора порога я превращаю полутоновое изображение в бинарное. **Операция `np.where` проверяет каждый пиксель: если его яркость больше `t`, присваивает 255, то есть белый цвет; иначе присваивает 0, то есть чёрный.** Тип `uint8` подходит для обычного восьмибитного изображения. Показываю результат и значение порога в заголовке. Важное ограничение глобального подхода — одно и то же значение порога применяется ко всем участкам картинки.

In [ ]:
B = np.where(gray > t, 255, 0).astype(np.uint8)
show(B, f"global Otsu, t = {t}")

> *Вывод по бинарному изображению.* Порог 97 разделил фотографию на чёрные пиксели (яркость 97 и ниже) и белые (яркость выше 97). Фигуры людей и некоторые яркие детали хорошо выделяются контрастом, однако полутона, тени и часть фактуры пропадают; яркий фон также становится белым. **Глобальный Оцу хорошо разделяет светлые и тёмные области, но один порог не всегда подходит для участков с разным освещением.**

2. Бинаризация методом Оцу локальная (с предварительным разбиением на фрагменты - по каждому своя бинаризация)

> В функции `local_otsu` сначала вычисляю общий порог для всего изображения — это запасной вариант. Потом с помощью `np.array_split` делю строки и столбцы на `n` частей и получаю сетку из `n × n` фрагментов. Для каждого фрагмента считаю стандартное отклонение яркостей: если оно меньше `min_std`, участок почти однородный, и локальный порог может быть неустойчивым, поэтому использую глобальный. Иначе рассчитываю отдельный порог методом Оцу. Затем бинаризую каждый фрагмент и собираю все части обратно. Функция возвращает итоговое изображение `B` и матрицу порогов `T`, по одному порогу на участок.

In [ ]:
def local_otsu(img, n, min_std=10):
    t_global, _ = otsu(histogram(img))
    B = np.zeros_like(img)
    T = np.zeros((n, n), dtype=int)

    rows = np.array_split(np.arange(img.shape[0]), n)
    cols = np.array_split(np.arange(img.shape[1]), n)
    for i, r in enumerate(rows):
        for j, c in enumerate(cols):
            tile = img[r[0]:r[-1] + 1, c[0]:c[-1] + 1]
            if tile.std() < min_std:
                t = t_global
            else:
                t, _ = otsu(histogram(tile))
            T[i, j] = t
            B[r[0]:r[-1] + 1, c[0]:c[-1] + 1] = np.where(tile > t, 255, 0)
    return B, T

In [ ]:
def draw_grid(img, n):
    vis = np.dstack([img] * 3)
    h, w = img.shape
    for k in range(1, n):
        x = round(k * w / n)
        y = round(k * h / n)
        vis = cv2.line(vis, (x, 0), (x, h), (255, 0, 0), 5)
        vis = cv2.line(vis, (0, y), (w, y), (255, 0, 0), 5)
    return vis

> **Здесь сравниваю локальный метод Оцу при разбиении изображения на сетки 2 на 2, 4 на 4 и 8 на 8.** При каждом значении `n` заново выполняется локальная бинаризация, после чего на результат накладываются границы фрагментов. Я смотрю, помогает ли более мелкая сетка выделять детали в разных областях и не возникают ли артефакты на стыках. Чем мельче фрагменты, тем меньше данных внутри каждого для надёжной оценки порога.

In [ ]:
for n in [2, 4, 8]:
    _B_local, _T = local_otsu(gray, n)
    show(draw_grid(_B_local, n), f"local Otsu, {n}x{n}")

> *Вывод по сравнению сеток.* На результате 2×2 заметно, что четыре большие области могут обрабатываться по-разному, но общий вид изображения остаётся относительно цельным. На сетках 4×4 и особенно 8×8 увеличивается число участков с отдельными порогами: некоторые детали становятся контрастнее, но на фоне и внутри фигур появляются резкие смены чёрного и белого, заметны блочные артефакты. Красные линии показывают границы участков. **Поэтому более мелкая сетка не обязательно означает лучший результат: она повышает локальную чувствительность, но может нарушать целостность изображения.**

3. Бинаризация методом Оцу иерархическая (выполняется с иерархическим разбиением каждой уже полученной  области, пока возможно разбивать). Квантование с использованием найденных порогов

> **В функции `hierarchical_otsu` я рекурсивно делю диапазон яркостей**. Сначала проверяю условия остановки: достигнута максимальная глубина, осталось меньше двух разных значений яркости или стандартное отклонение стало слишком маленьким. Если деление имеет смысл, метод Оцу находит порог внутри текущего диапазона. Далее функция вызывается отдельно для яркостей слева и справа от порога, а все найденные пороги объединяются в отсортированный список. Вторая функция, `quantize_by_thresholds`, применяет этот список к изображению: формирует интервалы яркости и заменяет каждую яркость внутри интервала средним значением его пикселей, рассчитанным по гистограмме. Получается адаптивное многоуровневое квантование.

In [ ]:
def hierarchical_otsu(hist, lo=0, hi=256, depth=0, max_depth=8, min_std=8):
    h = hist[lo:hi].astype(np.float64)
    if depth >= max_depth or np.count_nonzero(h) < 2:
        return []

    levels = np.arange(lo, hi)
    mean = (h * levels).sum() / h.sum()
    std = np.sqrt((h * (levels - mean) ** 2).sum() / h.sum())
    if std < min_std:
        return []

    t, _ = otsu(hist, lo, hi)
    left = hierarchical_otsu(hist, lo, t + 1, depth + 1, max_depth, min_std)
    right = hierarchical_otsu(hist, t + 1, hi, depth + 1, max_depth, min_std)
    return left + [t] + right


def quantize_by_thresholds(img, thresholds):
    hist = histogram(img)
    bounds = [0] + [t + 1 for t in thresholds] + [256]

    lut = np.zeros(256, dtype=np.uint8)
    for lo, hi in zip(bounds[:-1], bounds[1:]):
        h = hist[lo:hi]
        levels = np.arange(lo, hi)
        lut[lo:hi] = round((h * levels).sum() / h.sum()) if h.sum() else lo
    return cv2.LUT(img, lut)

> Теперь сравниваю иерархический метод при максимальной глубине рекурсии один, два и три. **Для каждого случая нахожу пороги, вывожу их численные значения и использую для квантования исходного изображения.** В заголовке показываю число полученных уровней, которое равно количеству порогов плюс один. Увеличение допустимой глубины может добавить уровни и сохранить больше оттенков, но не гарантирует их появление: рекурсия также останавливается по условиям разброса яркости.

In [ ]:
for depth in [1, 2, 3]:
    ts = hierarchical_otsu(hist, max_depth=depth)
    _Q = quantize_by_thresholds(gray, ts)
    print(depth, ts)
    show(_Q, f"hierarchical Otsu, depth = {depth}, levels = {len(ts) + 1}")

> *Вывод по порогам и изображениям.* При глубине 1 напечатан порог `[97]`, то есть получены 2 диапазона яркости. При глубине 2 пороги `[50, 97, 155]`, поэтому уровней 4. При глубине 3 пороги `[31, 50, 72, 97, 126, 155, 187]`, значит уровней 8. **Чем больше глубина разбиения, тем больше уровней яркости удалось получить: от 2 до 8.** На последовательных изображениях видны всё более плавные переходы и больше деталей лиц, одежды и фона. В отличие от обычной бинаризации, каждому диапазону присваивается его средняя яркость, а не просто 0 или 255. С увеличением глубины метод описывает распределение яркостей подробнее.

> **Теперь я увеличиваю максимально допустимую глубину иерархического разбиения до 16, но одновременно повышаю порог остановки `min_std` до 12**. Это означает, что диапазоны яркости с малым разбросом дальше не дробятся. В результате программа сама определяет, сколько уровней оставить. В полученном запуске это 8 уровней и 7 порогов: 50, 72, 97, 126, 155, 187 и 211. Сравниваю полученное полутоновое изображение с предыдущими результатами.

In [ ]:
thresholds = hierarchical_otsu(hist, max_depth=16, min_std=12)
Q_otsu = quantize_by_thresholds(gray, thresholds)
print(len(thresholds) + 1, thresholds)

show(Q_otsu, f"hierarchical Otsu, levels = {len(thresholds) + 1}")

>*Вывод по окончательному варианту.* Здесь напечатано `8 [50, 72, 97, 126, 155, 187, 211]`. **В финальном эксперименте алгоритм оставил 7 порогов и 8 уровней яркости, хотя максимальная глубина была задана равной 16.** Причина в том, что разбиение останавливается не только по глубине, но и когда разброс яркостей в группе становится меньше `min_std=12`. На изображении сохранено больше полутоновых деталей, чем при глобальной бинаризации. Набор порогов отличается от предыдущего 8-уровневого варианта также потому, что здесь изменено условие остановки: `min_std=12` вместо значения по умолчанию `8`.

> В завершающей ячейке я возвращаюсь к исходной гистограмме яркостей и отмечаю на ней найденные иерархическим методом пороги красными вертикальными линиями. По графику можно увидеть, в каких местах распределения алгоритм решил разделить яркости на группы. Так я визуально связываю результат квантования с данными исходного изображения. **Главное отличие от обычного квантования в том, что здесь границы групп зависят от гистограммы и не обязаны быть равномерными.**

In [ ]:
_fig, _ax = plt.subplots(figsize=(12, 3))
_ax.bar(np.arange(256), hist, width=1)
for _t in thresholds:
    _ax.axvline(_t, color="r")
plt.show()

> *Вывод по финальной гистограмме.* Красные вертикальные линии отмечают 7 найденных порогов: 50, 72, 97, 126, 155, 187 и 211. Они распределены неравномерно: алгоритм опирается на форму гистограммы и неоднородность яркостей, а не делит весь диапазон на равные интервалы. **При ручном квантовании границы уровней заданы равномерно, а при иерархическом Оцу они подбираются по гистограмме изображения.**